<a href="https://colab.research.google.com/github/Santiago-Echeverri-Arteaga/Fisica_Computacional_2/blob/master/curso_2026_2/03_redes_fundamentos/30_neurona_y_feedforward_desde_cero.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg"
       alt="Abrir en Colab"/>
</a>

# Neurona y propagación hacia adelante desde cero

**Pregunta guía:** ¿Qué calcula realmente una red densa?<br>
**Duración sugerida:** 4 horas.<br>
**Entorno:** CPU; datos incluidos o generados en memoria.

El orden de trabajo es siempre: problema → matemática → implementación
mínima → biblioteca → evaluación → interpretación física.


## De una neurona a una red

Una capa aplica una transformación afín y una no linealidad:

$$z^{(\ell)}=a^{(\ell-1)}W^{(\ell)}+b^{(\ell)},\qquad
a^{(\ell)}=\phi\!\left(z^{(\ell)}\right).$$

Para un lote de $N$ observaciones, $a^{(\ell-1)}$ es una matriz
$N\times n_{\ell-1}$ y $W^{(\ell)}$ tiene forma
$n_{\ell-1}\times n_\ell$. La no linealidad es indispensable: sin ella,
componer capas afines sigue produciendo una sola transformación afín.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

SEMILLA = 42
rng = np.random.default_rng(SEMILLA)

def relu(z):
    return np.maximum(0.0, z)

def tanh(z):
    return np.tanh(z)

class CapaDensa:
    def __init__(self, entradas, salidas, rng, activación=lambda z: z):
        # Inicialización de Xavier: varianza ~ 1/entradas.
        self.W = rng.normal(0, np.sqrt(1 / entradas), size=(entradas, salidas))
        self.b = np.zeros((1, salidas))
        self.activación = activación

    def __call__(self, a):
        self.z = a @ self.W + self.b
        self.a = self.activación(self.z)
        return self.a

X = rng.normal(size=(5, 3))
capa_1 = CapaDensa(3, 4, rng, relu)
capa_2 = CapaDensa(4, 1, rng)
salida = capa_2(capa_1(X))

print("X:", X.shape)
print("W1:", capa_1.W.shape, "a1:", capa_1.a.shape)
print("W2:", capa_2.W.shape, "salida:", salida.shape)


## Conteo de parámetros

Una capa con $n_{in}$ entradas y $n_{out}$ salidas contiene
$n_{in}n_{out}+n_{out}$ parámetros. Para 3→4→1 hay
$(3\times4+4)+(4\times1+1)=21$. Verificar formas y conteos antes de
entrenar previene muchos errores silenciosos.


In [ ]:
def contar_parámetros(*capas):
    return sum(capa.W.size + capa.b.size for capa in capas)

print("Parámetros:", contar_parámetros(capa_1, capa_2))
assert contar_parámetros(capa_1, capa_2) == 21


## Aproximación con características neuronales

Para visualizar potencia expresiva sin mezclar todavía backpropagation,
fijamos neuronas ocultas aleatorias $\tanh(w_jx+b_j)$ y calculamos por
mínimos cuadrados sólo los pesos de salida. Aumentar neuronas amplía el
espacio de funciones disponible, pero no garantiza generalización.


In [ ]:
x = np.linspace(-3, 3, 220)[:, None]
y = np.sin(2.2 * x[:, 0]) * np.exp(-0.12 * x[:, 0] ** 2)
índice = rng.permutation(len(x))
train, test = índice[:150], índice[150:]

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=True)
for ancho, ax in zip([3, 15, 80], axes):
    local_rng = np.random.default_rng(SEMILLA)
    W = local_rng.normal(size=(1, ancho))
    b = local_rng.uniform(-2, 2, size=(1, ancho))
    H_train = np.c_[np.ones(len(train)), np.tanh(x[train] @ W + b)]
    beta = np.linalg.lstsq(H_train, y[train], rcond=None)[0]
    H = np.c_[np.ones(len(x)), np.tanh(x @ W + b)]
    pred = H @ beta
    rmse = np.sqrt(np.mean((y[test] - pred[test]) ** 2))
    ax.plot(x, y, "k--", label="función")
    ax.plot(x, pred, label="red")
    ax.scatter(x[test], y[test], s=8, alpha=0.3)
    ax.set_title(f"{ancho} neuronas | RMSE={rmse:.3f}")
    ax.legend()
plt.show()


**Ejercicios:** demuestre algebraicamente por qué dos capas lineales se
reducen a una; cambie `tanh` por ReLU; calcule parámetros de una red
80→128→64→3; investigue qué ocurre al extrapolar fuera de $[-3,3]$.
